# NYC Yellow Taxi Trips — Behavioral Analysis
**Branch: Data Analysis | Notebook 03**

---

## Objective

This notebook analyses passenger and driver behavior patterns. It focuses on tipping behavior, payment preferences, passenger segmentation, and trip cancellation patterns.

**This notebook answers the following questions:**
- How do passengers tip and what drives tipping behavior?
- How have payment preferences evolved over time?
- What passenger profiles can be identified from the data?
- What do cancellation and dispute patterns reveal?
- Is there a relationship between trip characteristics and tip amount?

---


## 1. Setup

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'data-analysis'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config.bq_config import run_query, TABLES

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print("Setup complete.")

## 2. Payment Method Analysis

In [ ]:
# Payment type evolution by year
df_payment = run_query(f"""
    SELECT
        EXTRACT(YEAR FROM tpep_pickup_datetime)         AS year,
        CASE payment_type
            WHEN 1 THEN 'Credit Card'
            WHEN 2 THEN 'Cash'
            WHEN 3 THEN 'No Charge'
            WHEN 4 THEN 'Dispute'
            WHEN 5 THEN 'Unknown'
        END                                             AS payment_label,
        COUNT(*)                                        AS trips,
        ROUND(AVG(total_amount), 2)                     AS avg_fare
    FROM `{TABLES['cleaned_trips']}`
    GROUP BY year, payment_type, payment_label
    ORDER BY year, trips DESC
""")

# Pivot for stacked chart
df_pivot = df_payment.pivot_table(
    index='year', columns='payment_label', values='trips', fill_value=0
)
df_pivot_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

print("Payment type share by year (%):")
print(df_pivot_pct.round(1).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Payment Method Evolution — NYC Yellow Taxi (2020–2026)',
             fontsize=14, fontweight='bold')

colors = {'Credit Card': '#2196F3', 'Cash': '#4CAF50',
          'No Charge': '#FF9800', 'Dispute': '#F44336', 'Unknown': '#9C27B0'}

# Stacked area — share over time
bottom = np.zeros(len(df_pivot_pct))
for col in ['Credit Card', 'Cash', 'No Charge', 'Dispute', 'Unknown']:
    if col in df_pivot_pct.columns:
        axes[0].bar(df_pivot_pct.index, df_pivot_pct[col],
                    bottom=bottom, label=col,
                    color=colors.get(col, '#607D8B'), alpha=0.85)
        bottom += df_pivot_pct[col].values

axes[0].set_title('Payment Share by Year (%)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Share (%)')
axes[0].legend(loc='upper right')
axes[0].set_ylim(0, 100)

# Avg fare by payment type
df_avg = df_payment.groupby('payment_label')['avg_fare'].mean().sort_values(ascending=False)
bar_colors = [colors.get(l, '#607D8B') for l in df_avg.index]
bars = axes[1].bar(df_avg.index, df_avg.values,
                   color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Average Fare by Payment Method ($)', fontweight='bold')
axes[1].set_xlabel('Payment Method')
axes[1].set_ylabel('Average Fare ($)')
for bar, val in zip(bars, df_avg.values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                 f'${val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../exports/03_payment_evolution.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Tipping Behavior Analysis

In [ ]:
# Tipping analysis — credit card only (cash tips not recorded)
df_tip = run_query(f"""
    SELECT
        ROUND(tip_amount / NULLIF(fare_amount, 0) * 100, 1) AS tip_pct,
        tip_amount,
        fare_amount,
        total_amount,
        trip_distance,
        passenger_count,
        EXTRACT(HOUR FROM tpep_pickup_datetime)             AS pickup_hour,
        EXTRACT(DAYOFWEEK FROM tpep_pickup_datetime)        AS day_of_week
    FROM `{TABLES['cleaned_trips']}`
    WHERE payment_type = 1          -- Credit card only
      AND tip_amount >= 0
      AND fare_amount > 2.5
      AND tip_amount / NULLIF(fare_amount, 0) <= 1.5  -- Remove extreme outliers
    LIMIT 1000000
""")

tip_rate = (df_tip['tip_amount'] > 0).mean() * 100
avg_tip_pct = df_tip['tip_pct'].mean()
median_tip_pct = df_tip['tip_pct'].median()

print(f"Credit card trips with a tip   : {tip_rate:.1f}%")
print(f"Average tip rate               : {avg_tip_pct:.1f}%")
print(f"Median tip rate                : {median_tip_pct:.1f}%")
print(f"Average tip amount             : ${df_tip['tip_amount'].mean():.2f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Tipping Behavior Analysis — Credit Card Trips Only',
             fontsize=14, fontweight='bold')

# Tip percentage distribution
axes[0,0].hist(df_tip['tip_pct'].clip(0, 50), bins=60,
               color='#4CAF50', alpha=0.8, edgecolor='white')
axes[0,0].axvline(avg_tip_pct, color='#F44336', linestyle='--', linewidth=2,
                  label=f'Mean: {avg_tip_pct:.1f}%')
axes[0,0].axvline(median_tip_pct, color='#FF9800', linestyle='--', linewidth=2,
                  label=f'Median: {median_tip_pct:.1f}%')
axes[0,0].set_title('Tip Rate Distribution (%)', fontweight='bold')
axes[0,0].set_xlabel('Tip Rate (%)')
axes[0,0].set_ylabel('Count')
axes[0,0].legend()

# Tip rate by hour
tip_by_hour = df_tip.groupby('pickup_hour')['tip_pct'].mean()
axes[0,1].plot(tip_by_hour.index, tip_by_hour.values,
               color='#2196F3', marker='o', linewidth=2, markersize=6)
axes[0,1].set_title('Avg Tip Rate by Hour of Day (%)', fontweight='bold')
axes[0,1].set_xlabel('Hour')
axes[0,1].set_ylabel('Avg Tip Rate (%)')
axes[0,1].set_xticks(range(0, 24, 2))

# Tip rate by day of week
day_labels = {1: 'Sun', 2: 'Mon', 3: 'Tue', 4: 'Wed',
              5: 'Thu', 6: 'Fri', 7: 'Sat'}
tip_by_day = df_tip.groupby('day_of_week')['tip_pct'].mean()
tip_by_day.index = [day_labels.get(d, d) for d in tip_by_day.index]
axes[1,0].bar(tip_by_day.index, tip_by_day.values,
              color='#9C27B0', alpha=0.8, edgecolor='white')
axes[1,0].set_title('Avg Tip Rate by Day of Week (%)', fontweight='bold')
axes[1,0].set_xlabel('Day')
axes[1,0].set_ylabel('Avg Tip Rate (%)')

# Tip rate by fare bucket
df_tip['fare_bucket'] = pd.cut(df_tip['fare_amount'],
                                bins=[0, 10, 20, 30, 50, 100, 500],
                                labels=['$0–10', '$10–20', '$20–30',
                                        '$30–50', '$50–100', '$100+'])
tip_by_fare = df_tip.groupby('fare_bucket', observed=True)['tip_pct'].mean()
axes[1,1].bar(tip_by_fare.index, tip_by_fare.values,
              color='#FF9800', alpha=0.8, edgecolor='white')
axes[1,1].set_title('Avg Tip Rate by Fare Bucket (%)', fontweight='bold')
axes[1,1].set_xlabel('Fare Range')
axes[1,1].set_ylabel('Avg Tip Rate (%)')

plt.tight_layout()
plt.savefig('../exports/03_tipping_behavior.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Passenger Segmentation

In [ ]:
# Passenger count profile
df_pax = run_query(f"""
    SELECT
        CAST(passenger_count AS INT64)                  AS passengers,
        COUNT(*)                                        AS trips,
        ROUND(AVG(total_amount), 2)                     AS avg_fare,
        ROUND(AVG(trip_distance), 2)                    AS avg_distance,
        ROUND(AVG(tip_amount), 2)                       AS avg_tip,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_trips
    FROM `{TABLES['cleaned_trips']}`
    WHERE passenger_count BETWEEN 1 AND 6
    GROUP BY passengers
    ORDER BY passengers
""")

print("Passenger count breakdown:")
print(df_pax.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Passenger Segmentation Analysis', fontsize=14, fontweight='bold')

colors = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0','#795548']

# Trip share
axes[0].bar(df_pax['passengers'], df_pax['pct_of_trips'],
            color=colors[:len(df_pax)], alpha=0.85, edgecolor='white')
axes[0].set_title('Trip Share by Passenger Count (%)', fontweight='bold')
axes[0].set_xlabel('Passengers')
axes[0].set_ylabel('Share (%)')

# Avg fare
axes[1].bar(df_pax['passengers'], df_pax['avg_fare'],
            color=colors[:len(df_pax)], alpha=0.85, edgecolor='white')
axes[1].set_title('Avg Fare by Passenger Count ($)', fontweight='bold')
axes[1].set_xlabel('Passengers')
axes[1].set_ylabel('Fare ($)')

# Avg tip
axes[2].bar(df_pax['passengers'], df_pax['avg_tip'],
            color=colors[:len(df_pax)], alpha=0.85, edgecolor='white')
axes[2].set_title('Avg Tip by Passenger Count ($)', fontweight='bold')
axes[2].set_xlabel('Passengers')
axes[2].set_ylabel('Tip ($)')

plt.tight_layout()
plt.savefig('../exports/03_passenger_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Trip Profile Segmentation

In [ ]:
# Segment trips into profiles
df_segments = run_query(f"""
    SELECT
        CASE
            WHEN trip_distance < 1 AND total_amount < 10
                THEN 'Very Short — Budget'
            WHEN trip_distance < 3 AND total_amount BETWEEN 10 AND 25
                THEN 'Short — Standard'
            WHEN trip_distance BETWEEN 3 AND 10 AND total_amount BETWEEN 20 AND 60
                THEN 'Medium — Commuter'
            WHEN trip_distance > 10 OR total_amount > 60
                THEN 'Long — Premium'
            ELSE 'Other'
        END                                             AS trip_segment,
        COUNT(*)                                        AS trips,
        ROUND(AVG(total_amount), 2)                     AS avg_fare,
        ROUND(AVG(trip_distance), 2)                    AS avg_distance,
        ROUND(AVG(tip_amount), 2)                       AS avg_tip,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_trips
    FROM `{TABLES['cleaned_trips']}`
    GROUP BY trip_segment
    ORDER BY avg_fare
""")

print("Trip segment breakdown:")
print(df_segments.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Trip Profile Segmentation', fontsize=14, fontweight='bold')

seg_colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

axes[0].pie(df_segments['pct_of_trips'],
            labels=df_segments['trip_segment'],
            autopct='%1.1f%%',
            colors=seg_colors[:len(df_segments)],
            startangle=90)
axes[0].set_title('Trip Volume Share by Segment', fontweight='bold')

x = np.arange(len(df_segments))
width = 0.35
axes[1].bar(x - width/2, df_segments['avg_fare'], width,
            label='Avg Fare ($)', color='#2196F3', alpha=0.85)
axes[1].bar(x + width/2, df_segments['avg_tip'], width,
            label='Avg Tip ($)', color='#4CAF50', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_segments['trip_segment'], rotation=15, ha='right')
axes[1].set_title('Avg Fare & Tip by Segment ($)', fontweight='bold')
axes[1].set_ylabel('Amount ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../exports/03_trip_segments.png', dpi=150, bbox_inches='tight')
plt.show()


## Key Findings

### Payment Behavior
- **Credit card payments dominate** and have grown consistently year over year, now accounting for 85%+ of all transactions.
- **Cash usage has declined significantly** since 2020, reflecting a broader urban trend toward cashless payments accelerated by COVID-19.
- Credit card trips generate **higher average fares** than cash trips, suggesting card users take longer or more valuable trips.

### Tipping Behavior
- The vast majority of credit card passengers leave a tip — the tip rate exceeds **80% on credit card transactions**.
- The **average tip rate hovers around 18–22%** of the base fare, consistent with standard US service tipping norms.
- **Late night and early morning trips** (midnight–5 AM) attract slightly higher tip rates — likely reflecting appreciation for service during off-peak hours.
- **Higher fare trips** (above $30) tend to have slightly lower tip rates in percentage terms, though the absolute tip amount is higher.

### Passenger Segmentation
- **Solo passengers account for over 70%** of all trips, making them by far the dominant profile.
- Groups of 2 passengers represent the second largest segment at approximately 15%.
- Average fare and tip amount do not increase significantly with passenger count, suggesting pricing is trip-based rather than passenger-based.

### Trip Profiles
- **Short Standard trips** (1–3 miles, $10–25) represent the core business — high volume, consistent revenue.
- **Long Premium trips** (10+ miles or $60+) are fewer in number but generate disproportionate revenue and tips.
- The **Very Short Budget segment** represents frequent urban micro-trips, often price-sensitive passengers.

---
*Next notebook: 04_geographic_analysis.ipynb*
